In [19]:
##INSTALAR DEPENDENCIAS
!pip install requests pymongo python-dotenv

In [20]:
##IMPORTANCIONES Y CONEXCION A MONGO
import requests
import time
import random
import json
import os
from pymongo import MongoClient
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

print("xonectado a MongoDB Atlas!")

xonectado a MongoDB Atlas!


In [21]:
##VERIFICAR ROBOTS
resp = requests.get("https://api.lyrics.ovh/v1/Shakira/Waka Waka")
print(f"Status: {resp.status_code}")
print(f"API funcionando correctamente")

Status: 200
API funcionando correctamente


In [22]:
##BUENAS PRACTICAS
CONFIG = {
    "delay_min": 1,
    "delay_max": 3,
    "max_reintentos": 3,
    "timeout": 10,
    "max_errores": 100,
}

errores_consecutivos = 0

def request_seguro(url):
    global errores_consecutivos

    for intento in range(CONFIG["max_reintentos"]):
        try:
            time.sleep(random.uniform(CONFIG["delay_min"], CONFIG["delay_max"]))
            resp = requests.get(url, timeout=CONFIG["timeout"])

            if resp.status_code == 200:
                errores_consecutivos = 0
                return resp
            elif resp.status_code == 429:
                print(f" Rate limit, esperando 30 segundos...")
                time.sleep(30)
            else:
                print(f" Status {resp.status_code} en intento {intento+1}")

        except Exception as e:
            print(f" Error intento {intento+1}: {e}")
            time.sleep(5)

    errores_consecutivos += 1
    if errores_consecutivos >= CONFIG["max_errores"]:
        raise Exception(" Demasiados errores, detenido!")

    return None

print(" Listo")
print(f" Delay: {CONFIG['delay_min']}-{CONFIG['delay_max']} segundos")
print(f" Reintentos: {CONFIG['max_reintentos']}")
print(f" Para tras: {CONFIG['max_errores']} errores seguidos")

 Listo
 Delay: 1-3 segundos
 Reintentos: 3
 Para tras: 100 errores seguidos


In [23]:
##MAPEO ARTISTA -> GENERO (para guardar correctamente en MongoDB)
GENERO_POR_ARTISTA = {
    # Reggaeton / Latino urbano
    "Bad Bunny": "reggaeton", "Karol G": "reggaeton", "Daddy Yankee": "reggaeton",
    "J Balvin": "reggaeton", "Maluma": "reggaeton", "Ozuna": "reggaeton",
    "Nicky Jam": "reggaeton", "Anuel AA": "reggaeton", "Wisin": "reggaeton",
    "Don Omar": "reggaeton", "Pitbull": "reggaeton",
    # Pop latino
    "Shakira": "pop", "Juanes": "pop", "Maná": "pop", "Aventura": "pop",
    "Romeo Santos": "pop", "Marc Anthony": "pop", "Enrique Iglesias": "pop",
    "Carlos Vives": "pop", "Julieta Venegas": "pop", "Paulina Rubio": "pop",
    "Thalía": "pop", "Gloria Trevi": "pop", "Sin Bandera": "pop",
    "Jesse y Joy": "pop", "Ha Ash": "pop", "Reik": "pop", "Camila": "pop",
    "Alejandro Fernández": "pop", "Nek": "pop", "Marco Antonio Solís": "pop",
    "Intocable": "pop", "Fito Paez": "pop",
    # Rock internacional
    "Aerosmith": "rock", "Metallica": "rock", "Nirvana": "rock",
    "Pearl Jam": "rock", "Red Hot Chili Peppers": "rock", "Linkin Park": "rock",
    "Green Day": "rock", "Radiohead": "rock", "The Beatles": "rock",
    "The Rolling Stones": "rock", "Queen": "rock", "David Bowie": "rock",
    "AC DC": "rock", "Led Zeppelin": "rock", "Pink Floyd": "rock",
    "U2": "rock", "Depeche Mode": "rock", "The Cure": "rock",
    "Foo Fighters": "rock", "Audioslave": "rock", "Soundgarden": "rock",
    "Alice in Chains": "rock", "System of a Down": "rock",
    "Rage Against the Machine": "rock", "Muse": "rock",
    "Imagine Dragons": "rock", "Twenty One Pilots": "rock",
    "Fall Out Boy": "rock", "Panic at the Disco": "rock",
    # Rock latino
    "Soda Stereo": "rock", "Gustavo Cerati": "rock", "Charly Garcia": "rock",
    "Los Fabulosos Cadillacs": "rock", "Babasónicos": "rock",
    "Enanitos Verdes": "rock", "Los Prisioneros": "rock",
    "Café Tacvba": "rock", "Molotov": "rock", "Kinky": "rock",
    "Zoé": "rock", "Elefante": "rock",
    # Pop internacional
    "Lady Gaga": "pop", "Bruno Mars": "pop", "Taylor Swift": "pop",
    "Beyonce": "pop", "Rihanna": "pop", "Coldplay": "pop", "Ed Sheeran": "pop",
    "Adele": "pop", "The Weeknd": "pop", "Ariana Grande": "pop",
    "Billie Eilish": "pop", "Harry Styles": "pop", "Dua Lipa": "pop",
    "Olivia Rodrigo": "pop", "Justin Bieber": "pop", "Selena Gomez": "pop",
    "Katy Perry": "pop", "Michael Jackson": "pop", "Prince": "pop",
    "Gorillaz": "pop", "Oliver Tree": "pop",
    # Hip hop / Rap
    "Eminem": "hip hop", "Drake": "hip hop", "Post Malone": "hip hop",
    "Nicki Minaj": "hip hop", "Cardi B": "hip hop", "Kendrick Lamar": "hip hop",
    "Kanye West": "hip hop", "Jay Z": "hip hop", "Control Machete": "hip hop",
    # Reggae
    "Bob Marley": "reggae",
    # Country / Regional mexicano
    "Vicente Fernández": "country", "Joan Sebastian": "country",
    "Los Tucanes de Tijuana": "country", "Banda El Recodo": "country",
    "Pesado": "country",
}

##FUNCION PRINCIPAL SCRAPING
def get_letra(artista, cancion):
    url = f"https://api.lyrics.ovh/v1/{artista}/{cancion}"
    resp = request_seguro(url)
    if not resp:
        return None

    data = resp.json()
    if "lyrics" not in data or not data["lyrics"].strip():
        return None

    # Asignar genero desde el mapeo; si no esta en el dict queda como 'otros'
    genero = GENERO_POR_ARTISTA.get(artista, "otros")

    return {
        "titulo": cancion,
        "artista": artista,
        "letra": data["lyrics"].strip(),
        "fuente": "lyrics_ovh",
        "url_fuente": url,
        "fecha_recopilacion": datetime.now(),
        "genero": genero,
        "idioma": None,
        "embeddings": {"word2vec_avg": None, "beto_cls": None}
    }

In [24]:
##FUNCION PRINCIPAL SCRAPING
def get_letra(artista, cancion):
    url = f"https://api.lyrics.ovh/v1/{artista}/{cancion}"
    resp = request_seguro(url)
    if not resp:
        return None

    data = resp.json()
    if "lyrics" not in data or not data["lyrics"].strip():
        return None

    return {
        "titulo": cancion,
        "artista": artista,
        "letra": data["lyrics"].strip(),
        "fuente": "lyrics_ovh",
        "url_fuente": url,
        "fecha_recopilacion": datetime.now(),
        "genero": None,
        "idioma": None,
        "embeddings": {"word2vec_avg": None, "beto_cls": None}
    }

In [25]:
##OBTENER URLS DE LAS CANCIONES POR ARTISTAS

canciones_por_artista = {
    "Bad Bunny": ["Dakiti", "Titi Me Pregunto", "Me Porto Bonito", "Ojitos Lindos", "Moscow Mule",
                  "Yonaguni", "Efecto", "Neverita", "El Apagón", "Un Verano Sin Ti"],
    "Lady Gaga": ["Poker Face", "Bad Romance", "Just Dance", "Alejandro", "Born This Way",
                  "Shallow", "Telephone", "Edge of Glory", "Applause", "Million Reasons"],
    "Gorillaz": ["Feel Good Inc", "Clint Eastwood", "Stylo", "On Melancholy Hill", "DARE",
                 "Rhinestone Eyes", "Empire Ants", "Tranz", "Humanz", "Momentary Bliss"],
    "Bruno Mars": ["Just the Way You Are", "Grenade", "Locked Out of Heaven", "Uptown Funk",
                   "That's What I Like", "Leave the Door Open", "The Lazy Song", "Treasure", "When I Was Your Man", "Versace on the Floor"],
    "Oliver Tree": ["Hurt", "Life Goes On", "Cowboy Tears", "Let Me Down", "Jerk",
                    "Cash Machine", "Waste It on Me", "Again and Again", "When I'm Down", "Joke's on You"],
    "Shakira": ["Waka Waka", "Hips Don't Lie", "La Tortura", "Ciega Sordomuda", "Suerte",
                "She Wolf", "Whenever Wherever", "Loca", "Rabiosa", "Can't Remember to Forget You"],
    "Karol G": ["Tusa", "Bichota", "Mamiii", "Provenza", "200 Copas",
                "El Makinon", "Ay Dios Mío", "Location", "Senorita", "Pineapple"],
    "Daddy Yankee": ["Gasolina", "Con Calma", "Shaky Shaky", "Dura", "Rompe",
                     "Limbo", "Lovumba", "Pose", "Que Tire Pa Lante", "Boom Boom"],
    "J Balvin": ["Mi Gente", "Safari", "Ginza", "Ay Vamos", "6AM",
                 "Ambiente", "Bonita", "Reggaeton", "X", "Con Calma"],
    "Maluma": ["Hawái", "Felices los 4", "Corazon", "Cuatro Babys", "Marry Me",
               "Borró Cassette", "La Temperatura", "El Perdedor", "Junio", "GPS"],
    "Ozuna": ["Taki Taki", "Caramelo", "Baila Baila Baila", "Una Flor", "El Farsante",
              "Quiero Repetir", "Criminal", "La Modelo", "Vacía Sin Ti", "Después"],
    "Juanes": ["La Camisa Negra", "A Dios Le Pido", "Me Enamora", "Fotografía",
               "Es Por Ti", "Para Tu Amor", "Mala Gente", "Tres", "Volverte a Ver", "Nada Valgo Sin Tu Amor"],
    "Maná": ["Oye Mi Amor", "En El Muelle de San Blas", "Labios Compartidos", "Rayando el Sol",
             "Vivir Sin Aire", "Si No Te Hubieras Ido", "De Pies a Cabeza", "Mariposa Traicionera", "Bendita tu Luz", "Amor es Combate"],
    "Aventura": ["Obsesión", "Por Un Segundo", "Ella y Yo", "Su Veneno",
                 "Dile al Amor", "Los Infieles", "Todo Tiene Su Hora", "La Boda", "No", "Un Chi Chi"],
    "Romeo Santos": ["Propuesta Indecente", "Eres Mía", "Canción del Año", "Fui a Jamaica",
                     "Odio", "Dulce", "Yo También", "Hilito", "Promise", "Debate de 4"],
    "Don Omar": ["Danza Kuduro", "Taboo", "Dutty Love", "Virtual Diva",
                 "Hasta Abajo", "Conteo", "Pobre Diabla", "Salio el Sol", "Los Bandoleros", "Sexy Robotica"],
    "Marc Anthony": ["Vivir Mi Vida", "Flor Pálida", "Y Hubo Alguien", "Tu Amor Me Hace Bien",
                     "Valió la Pena", "Preciosa", "Aguanile", "Nadie Como Ella", "Palabras del Alma", "La Celos"],
    "Enrique Iglesias": ["Hero", "Bailamos", "Be With You", "Escape", "I Like It",
                         "Subeme la Radio", "El Perdón", "Duele el Corazon", "Experiencia Religiosa", "Dirty Dancer"],
    "Carlos Vives": ["La Bicicleta", "Volví a Nacer", "Robarte un Beso", "Bailar Contigo",
                     "Fruta Fresca", "Carito", "Cuando Nos Volvamos a Ver", "Corazón Profundo", "El Mar de Sus Ojos", "Déjame Entrar"],
    "Eminem": ["Lose Yourself", "Slim Shady", "Without Me", "Not Afraid", "Love the Way You Lie",
               "Rap God", "The Real Slim Shady", "Stan", "Mockingbird", "Beautiful"],
    "Drake": ["God's Plan", "Hotline Bling", "One Dance", "In My Feelings", "Started From the Bottom",
              "Hold On We're Going Home", "Nice for What", "Controlla", "Forever", "Nonstop"],
    "Taylor Swift": ["Shake It Off", "Blank Space", "Love Story", "Bad Blood", "You Belong With Me",
                     "Anti-Hero", "Wildest Dreams", "Style", "Delicate", "Cardigan"],
    "Beyonce": ["Halo", "Crazy in Love", "Single Ladies", "Irreplaceable", "Lemonade",
                "Formation", "Love on Top", "Drunk in Love", "Sorry", "Partition"],
    "Rihanna": ["Umbrella", "Diamonds", "We Found Love", "Stay", "Work",
                "Love the Way You Lie", "Only Girl", "What's My Name", "Needed Me", "Disturbia"],
    "Coldplay": ["Yellow", "The Scientist", "Fix You", "Viva la Vida", "Clocks",
                 "Paradise", "Speed of Sound", "A Sky Full of Stars", "Magic", "In My Place"],
    "Ed Sheeran": ["Shape of You", "Thinking Out Loud", "Perfect", "Photograph", "Castle on the Hill",
                   "Bad Habits", "Shivers", "Overpass Graffiti", "Galway Girl", "Don't"],
    "Adele": ["Hello", "Rolling in the Deep", "Someone Like You", "Set Fire to the Rain",
              "Skyfall", "Send My Love", "Water Under the Bridge", "When We Were Young", "Easy On Me", "Turning Tables"],
    "The Weeknd": ["Blinding Lights", "Starboy", "Can't Feel My Face", "The Hills",
                   "Save Your Tears", "Earned It", "Call Out My Name", "Heartless", "Often", "In Your Eyes"],
    "Ariana Grande": ["Thank U Next", "7 Rings", "Problem", "Into You", "No Tears Left to Cry",
                      "God is a Woman", "Break Free", "Side to Side", "Positions", "Rain on Me"],
    "Post Malone": ["Rockstar", "Congratulations", "Sunflower", "Circles", "Better Now",
                    "White Iverson", "Go Flex", "Psycho", "Stay", "I Fall Apart"],
    "Billie Eilish": ["Bad Guy", "Happier Than Ever", "Ocean Eyes", "Bury a Friend", "When the Party's Over",
                      "Therefore I Am", "Your Power", "NDA", "Lovely", "Everything I Wanted"],
    "Harry Styles": ["Watermelon Sugar", "Adore You", "Lights Up", "Golden", "Sign of the Times",
                     "As It Was", "Late Night Talking", "Matilda", "Music for a Sushi Restaurant", "Canyon Moon"],
    "Dua Lipa": ["Levitating", "Don't Start Now", "New Rules", "Physical", "Break My Heart",
                 "One Kiss", "Electricity", "Be the One", "IDGAF", "Love Again"],
    "Olivia Rodrigo": ["drivers license", "good 4 u", "deja vu", "brutal", "traitor",
                       "happier", "favorite crime", "enough for you", "1 step forward", "hope ur ok"],
    "Justin Bieber": ["Baby", "Love Yourself", "Sorry", "What Do You Mean", "Intentions",
                      "Peaches", "Stay", "Ghost", "Anyone", "Lonely"],
    "Selena Gomez": ["Come and Get It", "Good for You", "Same Old Love", "Hands to Myself",
                     "Lose You to Love Me", "Rare", "Look at Her Now", "Back to You", "Wolves", "Fetish"],
    "Katy Perry": ["Roar", "Firework", "Dark Horse", "California Gurls", "Teenage Dream",
                   "Last Friday Night", "Wide Awake", "Part of Me", "ET", "Hot N Cold"],
    "Nicki Minaj": ["Super Bass", "Anaconda", "Starships", "Bang Bang", "Only",
                    "Feeling Myself", "Chun-Li", "Barbie Dreams", "Good Form", "Moment 4 Life"],
    "Cardi B": ["Bodak Yellow", "I Like It", "Money", "WAP", "Bartier Cardi",
                "Ring", "Be Careful", "Drip", "Taki Taki", "Press"],
    "Kendrick Lamar": ["HUMBLE", "DNA", "Swimming Pools", "Alright", "Bitch Don't Kill My Vibe",
                       "King Kunta", "Money Trees", "Poetic Justice", "Loyalty", "Love"],
    "Kanye West": ["Stronger", "Gold Digger", "Heartless", "All Falls Down", "Jesus Walks",
                   "Power", "Runaway", "Black Skinhead", "Flashing Lights", "Bound 2"],
    "Jay Z": ["Empire State of Mind", "99 Problems", "Izzo", "Big Pimpin", "Run This Town",
              "Holy Grail", "On to the Next One", "Young Forever", "Dirt Off Your Shoulder", "Hard Knock Life"],
    "Nicky Jam": ["El Perdón", "Hasta el Amanecer", "X", "Ran Ran",
                  "Loco Contigo", "Te Busco", "Quiero Conocerte", "Voy a Beber", "Íntimo", "Travesuras"],
    "Anuel AA": ["China", "Secreto", "Ella Quiere Beber", "Brindemos",
                 "Hipócrita", "Keii", "Sola", "Relación", "Amanece", "Mayor Que Yo 3"],
    "Wisin": ["Loco", "Algo Me Gusta de Ti", "Adrenalina", "Quisiera Alejarse",
              "Que Viva la Vida", "Tu Foto", "No Me Des Celos", "Follow the Leader", "Manos al Aire", "Noche de Sexo"],
    "Pitbull": ["Give Me Everything", "International Love", "Feel This Moment", "Timber",
                "Don't Stop the Party", "Time of Our Lives", "Wild Wild Love", "Back in Time", "Hey Baby", "Rain Over Me"],
    "Julieta Venegas": ["Me Voy", "Lento", "Limón y Sal", "Eres Para Mí",
                        "Bien o Mal", "Andar Conmigo", "Algo Está Cambiando", "Nada", "El Presente", "Sería Feliz"],
    "Café Tacvba": ["La Ingrata", "Eres", "Quiero Ver", "Cómo Te Extraño Mi Querida",
                    "Olvídame y Pega la Vuelta", "Chilanga Banda", "Mediodía", "El Aparato", "Volver a Comenzar", "Encantamiento Guau Guau"],
    "Molotov": ["Frijolero", "Gimme Tha Power", "Hit Me", "Voto Latino",
                "Puto", "Apocalypshit", "Amateur", "Monstruo", "Rastaman", "Que No Te Haga Bobo Ponchis Ponchis"],
    "Soda Stereo": ["De Música Ligera", "Persiana Americana", "Nada Personal", "Cuando Pase el Temblor",
                    "En la Ciudad de la Furia", "Signos", "Primavera Cero", "Trátame Suavemente", "Zoom", "Juego de Seducción"],
    "Gustavo Cerati": ["Crimen", "Lago en el Cielo", "Fantasma", "Sudestada",
                       "Vivo", "Fuerza Natural", "Adiós", "Beautiful", "Sal", "He Visto a Lucy"],
    "Fito Paez": ["El Amor Después del Amor", "Tumbas de la Gloria", "11 y 6", "Yo Vengo a Ofrecer mi Corazón",
                  "Cable a Tierra", "Garganta con Arena", "Un Vestido y un Amor", "Mariposa Tecknicolor", "Decisión", "Brillante sobre el Mic"],
    "Charly Garcia": ["No Bombardeen Buenos Aires", "Nos Siguen Pegando Abajo", "Cerca de la Revolución",
                      "Los Dinosaurios", "Demoliendo Hoteles", "Cuál Es", "Fanky", "Promesas Sobre el Bidet", "Filosofía Barata", "Inconsciente Colectivo"],
    "Los Fabulosos Cadillacs": ["Matador", "Mal Bicho", "El Satánico Dr. Cadillac", "Siguiendo la Luna",
                                "Vasos Vacíos", "Carnaval Toda la Vida", "Mi Novia Se Cayó en un Pozo Ciego", "Padre Nuestro", "Gitana", "El León"],
    "Babasónicos": ["Putita", "Yegua", "Fizz", "Tóxico",
                    "Deléctrico", "Irresponsables", "Microdancing", "Los Calientes", "Deja Vu", "Como Conseguir Chicas"],
    "Enanitos Verdes": ["Lamento Boliviano", "La Muralla Verde", "Cuánto Poder", "Amame",
                        "Hasta Que Amanezca", "Tu Cárcel", "Quiero Más", "Los Prisioneros", "Bonito", "Melaza"],
    "Los Prisioneros": ["La Voz de los 80", "We Are Sudamerican Rockers", "El Baile de los Que Sobran",
                        "Corazones Rojos", "Tren al Sur", "Por Eso los Pobres Van al Infierno", "Nunca Quedas Mal con Nadie", "Sexo", "Maldito Sudaca", "Estrechez de Corazón"],
    "Nek": ["Si Me Dejas Ahora", "Una Notte Che Non Passa", "Laura Non C'é", "Lasciami",
            "Cuéntame", "Il Tempo Non Torna Più", "Almeno Stavolta", "Tu Solo Tu", "Sei Grande", "Lei Non Sa"],
    "Alejandro Fernández": ["Quiero Que Vuelvas", "Loco", "Mátenme Porque Me Muero",
                             "Como Quien Pierde una Estrella", "Quiero Que Sepas", "Prohibido", "Que Lo Diga", "Nube Viajera", "Sé Que Te Duele", "Amor de los Dos"],
    "Vicente Fernández": ["Volver Volver", "El Rey", "Lástima que Seas Ajena", "Por Tu Maldito Amor",
                          "La Ley del Monte", "Acá Entre Nos", "Mujeres Divinas", "Chilindrina", "A Pesar de Todo", "Estos Celos"],
    "Joan Sebastian": ["Tatuajes", "Secreto de Amor", "Eso y Más", "Bandido",
                       "La Malagradecida", "Ántrax", "Me Gustas", "Mal Hombre", "El Quijote", "Ya Lo Sé"],
    "Marco Antonio Solís": ["Si No Te Hubieras Ido", "La Venia Bendita", "Que No Quede Huella",
                             "Tú Me Volviste a Enamorar", "El Pasado", "En Tus Manos", "Como Fui a Enamorarme de Ti", "Morenita", "Te Quiero", "El Amor de Mi Vida"],
    "Intocable": ["Romántico", "Así es el Amor", "No Mereces Mis Lágrimas", "Prometiste",
                  "Me Pasé", "Te Quiero", "Amor de Dios", "No Volvere", "Contigo", "Quiero Más"],
    "Los Tucanes de Tijuana": ["La Chona", "El Tigrillo", "El Corrido del Chapo", "Mis Tres Animales",
                                "Una Aventura", "El Toro Grande", "El Centenario", "Don Corrido", "La Mafia del Diablo", "Mis Memorias"],
    "Banda El Recodo": ["El Sinaloense", "La Buena Forma", "Corazón de Papel", "Y Llegaste Tú",
                        "El Mechón", "Las Isabeles", "Que Me Entierren con la Banda", "Amorcito Corazón", "Popurrí Grupero", "El Sinaloa"],
    "Pesado": ["Eres un Encanto", "Cuando el Amor se Va", "La Bamba de Jalisco",
               "Directo al Corazón", "Quiero Amanecer", "Se Fue", "Que Se Pudra", "Tu Recuerdo", "Sigue Lloviendo", "Paloma Negra"],
    "Control Machete": ["Andamos Armados", "Sí Señor", "Comprendes Mendes",
                        "¿Qué Es Lo Que Quiero?", "Héroes del Norte", "Muévelo", "El Genio", "Humanos Mexicanos", "Pelea Tu Pelea", "El Mas Chingón"],
    "Kinky": ["Soun Tha Mi Primer Amor", "De Fondo Musical", "Más", "Cornman",
              "Leche", "Me Cae Bien Tu Perro", "Beso", "Panza", "A Donde Van los Muertos", "Coqueta"],
    "Zoé": ["Noche de Brujas", "Soñé", "Luna", "Paula", "Vía Láctea",
            "Reptilectric", "Azul", "Venona", "Musical", "Love"],
    "Elefante": ["Sabes", "Tú", "Falsas Esperanzas", "Me Voy",
                 "Mala Mujer", "Eres Todo", "Valió la Pena", "Contigo", "Corazón", "Muriendo Lento"],
    "Reik": ["Yo Quisiera", "Noviembre Sin Ti", "Qué Vida la Mía", "Creo en Ti",
             "Me Niego", "Sin Ti No Puedo", "Un Año", "Gracias a Ti", "Patience", "Ahora"],
    "Camila": ["Mientes", "Todo Cambió", "Alejate de Mi", "Bésame",
               "Aléjate de Mi", "Herida", "Por Favor", "Abrázame", "Se Fue", "Coleccionista de Canciones"],
    "Sin Bandera": ["Que Me Alcance la Vida", "Amor Real", "Donde Estarás",
                    "De Viaje", "Sirena", "Lo Que Sangra", "A Partir de Hoy", "Como Quisiera", "Hay Momentos", "Incondicional"],
    "Jesse y Joy": ["Corre", "¡Corre!", "Tanto", "Espacio Sideral",
                    "No Soy una de Esas", "Mañana", "Me Voy", "Ecos de Amor", "Stupid", "Un Besito Más"],
    "Ha Ash": ["100 Años", "Lo Que Tenemos", "Te Fuiste", "Perdón Perdón",
               "Primera Fila", "Que Me Faltas Tú", "No Me Pidas Que No Te Quiera", "Un Millón de Años", "Quiero Más", "Tú Siempre Tú"],
    "Paulina Rubio": ["Te Quise Tanto", "La Chica Dorada", "Yo No Soy Esa Mujer", "Me Gustas Tanto",
                      "Algo Tienes", "Ni Una Sola Palabra", "Causa y Efecto", "Boys Will Be Boys", "Baila la Guaracha", "En Todos Los Ángulos"],
    "Thalía": ["Amor a la Mexicana", "No Me Enseñaste", "Piel Morena", "Arrasando",
               "Equivocada", "Aún Recuerdo", "Hola", "A Quien le Importa", "Cerca de Ti", "Manías"],
    "Gloria Trevi": ["Pelo Suelto", "Dr. Psiquiatra", "Todos Me Miran", "Me Siento Tan Sola",
                     "Zapatos Viejos", "No Querías Lastimarme", "Cinco Minutos", "Habla el Corazón", "Vestida de Azúcar", "Ábranse"],
    "Aerosmith": ["Dream On", "Sweet Emotion", "Walk This Way", "I Don't Want to Miss a Thing",
                  "Cryin", "Crazy", "Amazing", "Janie's Got a Gun", "Love in an Elevator", "Angel"],
    "Metallica": ["Enter Sandman", "Master of Puppets", "Nothing Else Matters", "One",
                  "Fade to Black", "Seek and Destroy", "The Unforgiven", "Wherever I May Roam", "Sad but True", "For Whom the Bell Tolls"],
    "Nirvana": ["Smells Like Teen Spirit", "Come as You Are", "Lithium", "Heart-Shaped Box",
                "In Bloom", "Polly", "Rape Me", "All Apologies", "About a Girl", "Pennyroyal Tea"],
    "Pearl Jam": ["Jeremy", "Even Flow", "Alive", "Black", "Better Man",
                  "Last Kiss", "Given to Fly", "Yellow Ledbetter", "Nothingman", "Just Breathe"],
    "Red Hot Chili Peppers": ["Under the Bridge", "Californication", "Scar Tissue", "Give It Away",
                               "Otherside", "By the Way", "Can't Stop", "Soul to Squeeze", "Road Trippin", "Dani California"],
    "Linkin Park": ["In the End", "Numb", "Crawling", "Somewhere I Belong", "Breaking the Habit",
                    "What I've Done", "Faint", "New Divide", "Shadow of the Day", "Waiting for the End"],
    "Green Day": ["Basket Case", "Boulevard of Broken Dreams", "Wake Me Up When September Ends",
                  "American Idiot", "Good Riddance", "Holiday", "21 Guns", "When I Come Around", "Longview", "Brain Stew"],
    "Radiohead": ["Creep", "Karma Police", "Fake Plastic Trees", "No Surprises", "High and Dry",
                  "Street Spirit", "Paranoid Android", "Exit Music", "Lucky", "Pyramid Song"],
    "The Beatles": ["Hey Jude", "Let It Be", "Come Together", "Yesterday", "Something",
                    "Blackbird", "Here Comes the Sun", "Eleanor Rigby", "Help", "In My Life"],
    "The Rolling Stones": ["Paint It Black", "Satisfaction", "Sympathy for the Devil", "Start Me Up",
                           "Gimme Shelter", "Wild Horses", "Angie", "Jumping Jack Flash", "Miss You", "Brown Sugar"],
    "Queen": ["Bohemian Rhapsody", "We Will Rock You", "We Are the Champions", "Somebody to Love",
              "Don't Stop Me Now", "Another One Bites the Dust", "Under Pressure", "Radio Ga Ga", "Killer Queen", "Fat Bottomed Girls"],
    "David Bowie": ["Space Oddity", "Heroes", "Ziggy Stardust", "Changes", "Let's Dance",
                    "Life on Mars", "Golden Years", "Modern Love", "China Girl", "Fame"],
    "Michael Jackson": ["Thriller", "Billie Jean", "Beat It", "Black or White", "Man in the Mirror",
                        "Smooth Criminal", "Don't Stop Till You Get Enough", "Rock With You", "Bad", "Earth Song"],
    "Prince": ["Purple Rain", "When Doves Cry", "Kiss", "Little Red Corvette", "Sign O the Times",
               "Let's Go Crazy", "Raspberry Beret", "1999", "Around the World in a Day", "Cream"],
    "Bob Marley": ["No Woman No Cry", "Redemption Song", "One Love", "Three Little Birds",
                   "Buffalo Soldier", "Is This Love", "Could You Be Loved", "Jamming", "Exodus", "Waiting in Vain"],
    "AC DC": ["Highway to Hell", "Back in Black", "Thunderstruck", "You Shook Me All Night Long",
              "TNT", "Hells Bells", "Dirty Deeds Done Dirt Cheap", "Rock and Roll Ain't Noise Pollution", "For Those About to Rock", "Shoot to Thrill"],
    "Led Zeppelin": ["Stairway to Heaven", "Whole Lotta Love", "Black Dog", "Kashmir",
                     "Rock and Roll", "Since I've Been Loving You", "Immigrant Song", "Ramble On", "Tangerine", "Going to California"],
    "Pink Floyd": ["Wish You Were Here", "Comfortably Numb", "Another Brick in the Wall",
                   "Money", "Time", "Breathe", "Hey You", "Shine On You Crazy Diamond", "The Wall", "Brain Damage"],
    "U2": ["One", "With or Without You", "Beautiful Day", "Where the Streets Have No Name",
           "Sunday Bloody Sunday", "I Still Haven't Found What I'm Looking For", "Pride", "New Year's Day", "Elevation", "Vertigo"],
    "Depeche Mode": ["Personal Jesus", "Enjoy the Silence", "Policy of Truth", "Just Can't Get Enough",
                     "People Are People", "Master and Servant", "Everything Counts", "Never Let Me Down Again", "Precious", "Wrong"],
    "The Cure": ["Lovesong", "Boys Don't Cry", "Close to Me", "Friday I'm in Love",
                 "In Between Days", "Pictures of You", "Fascination Street", "Disintegration", "Lullaby", "The Caterpillar"],
    "Depeche Mode": ["Personal Jesus", "Enjoy the Silence", "Policy of Truth", "Just Can't Get Enough",
                     "People Are People", "Master and Servant", "Everything Counts", "Never Let Me Down Again", "Precious", "Wrong"],
    "Foo Fighters": ["Everlong", "Best of You", "Learn to Fly", "The Pretender",
                     "All My Life", "Times Like These", "Walk", "Rope", "These Days", "Long Road to Ruin"],
    "Audioslave": ["Like a Stone", "Show Me How to Live", "Be Yourself", "Cochise",
                   "Doesn't Remind Me", "Exploder", "Set It Off", "Your Time Has Come", "Revelations", "Original Fire"],
    "Soundgarden": ["Black Hole Sun", "Spoonman", "Fell on Black Days", "Burden in My Hand",
                    "The Day I Tried to Live", "Like Suicide", "Rusty Cage", "Mailman", "My Wave", "Pretty Noose"],
    "Alice in Chains": ["Would", "Rooster", "Down in a Hole", "Them Bones",
                        "No Excuses", "Angry Chair", "Rain When I Die", "Got Me Wrong", "Nutshell", "I Stay Away"],
    "System of a Down": ["Chop Suey", "Toxicity", "B.Y.O.B", "Aerials",
                         "Lonely Day", "Hypnotize", "Cigaro", "Prison Song", "Spiders", "Question"],
    "Rage Against the Machine": ["Killing in the Name", "Bulls on Parade", "Guerrilla Radio",
                                  "Wake Up", "Bombtrack", "Testify", "Renegades", "Ashes in the Fall", "No Shelter", "Sleep Now in the Fire"],
    "Muse": ["Supermassive Black Hole", "Uprising", "Madness", "Starlight", "Time is Running Out",
             "Knights of Cydonia", "Resistance", "Hysteria", "Map of the Problematique", "Neutron Star Collision"],
    "Imagine Dragons": ["Radioactive", "Demons", "Believer", "Thunder", "Enemy",
                        "Natural", "Whatever It Takes", "Monsters", "Bad Liar", "Follow You"],
    "Twenty One Pilots": ["Stressed Out", "Ride", "Heathens", "Tear in My Heart",
                          "Jumpsuit", "Morph", "Chlorine", "Levitate", "Legend", "My Blood"],
    "Fall Out Boy": ["Sugar We're Going Down", "Thnks fr th Mmrs", "Dance Dance", "I Don't Care",
                     "My Songs Know What You Did in the Dark", "Centuries", "Uma Thurman", "Alone Together", "The Phoenix", "Young Volcanoes"],
    "Panic at the Disco": ["I Write Sins Not Tragedies", "Nine in the Afternoon", "This is Gospel",
                            "Death of a Bachelor", "High Hopes", "Emperor's New Clothes", "LA Devotee", "Victorious", "Ready to Go", "Girls Girls Boys"],
}

print(f"{len(canciones_por_artista)} artistas")
print(f"{sum(len(v) for v in canciones_por_artista.values())} canciones en lista")

41 artistas
410 canciones en lista


In [26]:
##GUARDAR Y SEGUIR
CHECKPOINT_FILE = "../data/processed/scraping_checkpoint.json"

def guardar_checkpoint(artista_actual, cancion_actual, total_insertadas):
    checkpoint = {
        "artista_actual": artista_actual,
        "cancion_actual": cancion_actual,
        "total_insertadas": total_insertadas,
        "ultima_actualizacion": datetime.now().isoformat()
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f)

def cargar_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            contenido = f.read().strip()
            if not contenido:
                return None
            return json.loads(contenido)
    return None

checkpoint = cargar_checkpoint()
if checkpoint:
    total_insertadas = checkpoint["total_insertadas"]
    print(f" Checkpoint encontrado!")
    print(f"   Último artista: {checkpoint['artista_actual']}")
    print(f"   Canciones insertadas: {total_insertadas}")
else:
    total_insertadas = coleccion.count_documents({"fuente": "lyrics_ovh"})
    print(f" Empezando. Ya hay {total_insertadas} canciones de lyrics.ovh en MongoDB")

 Checkpoint encontrado!
   Último artista: Bad Bunny
   Canciones insertadas: 850


In [27]:
##SCRAPING MASIVO Y GUARDAR EN MONGO
META = 1000
total_errores = 0

try:
    for artista, canciones in canciones_por_artista.items():
        if total_insertadas >= META:
            break

        print(f"\n Procesando: {artista}")

        for cancion in canciones:
            if total_insertadas >= META:
                break

            # Verificar duplicado
            if coleccion.find_one({"artista": artista, "titulo": cancion, "fuente": "lyrics_ovh"}):
                print(f" Ya existe: {cancion}")
                continue

            doc = get_letra(artista, cancion)

            if doc:
                coleccion.insert_one(doc)
                total_insertadas += 1
                print(f"[{total_insertadas}/{META}] {cancion}")
            else:
                total_errores += 1
                print(f" No encontrada: {cancion}")

            # Guardar checkpoint cada 50
            if total_insertadas % 50 == 0 and total_insertadas > 0:
                guardar_checkpoint(artista, cancion, total_insertadas)
                print(f" Checkpoint guardado: {total_insertadas} canciones")

except Exception as e:
    print(f"\n  Detenido: {e}")
    guardar_checkpoint(artista, cancion, total_insertadas)

finally:
    print(f"\n Sesión finalizada!")
    print(f" Canciones insertadas: {total_insertadas}")
    print(f" Errores: {total_errores}")
    print(f" Total en MongoDB: {coleccion.count_documents({})}")


 Procesando: Elefante

  Detenido: SSL handshake failed: ac-c1alydh-shard-00-00.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-c1alydh-shard-00-01.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-c1alydh-shard-00-02.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69c5d16924158a6cb2f4a425, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-c1alydh-shard-00-00.xdk6hlm.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-

KeyboardInterrupt: 

In [10]:
##CERRAR CONEXCIONES
client.close()
print("Conexión cerrada")

Conexión cerrada


In [21]:
##comprobar urls

import requests
resp = requests.get("https://api.lyrics.ovh/v1/Shakira/Waka Waka")
print(resp.status_code)
print(resp.json())

200
{'lyrics': "Oooeeeeeeeeeeeeeeeehh\n\nYou're a good soldier\nChoosing your battles\nPick yourself up\nAnd dust yourself off\nGet back in the saddle\n\nYou're on the front line\nEveryone's watching\nYou know it's serious\nWe're getting closer\nThis isn't over\n\nThe pressure's on; you feel it\nBut you got it all; believe it\nWhen you fall, get up, oh oh\nAnd if you fall, get up, eh eh\nTsamina mina zangalewa\nCause this is Africa\n\nTsamina mina eh eh\nWaka waka eh eh\nTsamina mina zangalewa\nThis time for Africa\n\nListen to your god\nthis is our motto\nYour time to shine\nDon't wait in line\nY vamos por todo\n\nPeople are raising their expectations\nGo on and feel it\nThis is your moment\nNo hesitation\n\nToday's your day\nI feel it\nYou paved the way,\nBelieve it\nIf you get down\nGet up oh, oh\nWhen you get down,\nGet up eh, eh\nTsamina mina zangalewa\nThis time for Africa\n\nTsamina mina eh eh\nWaka waka eh eh\nTsamina mina zangalewa\nAnawa aa\n\nTsamina mina eh eh\nWaka waka eh

In [44]:
doc = coleccion.find_one({"fuente": "lyrics_ovh"})
print(doc)

{'_id': ObjectId('69c0645c21e0de0e7991a34f'), 'titulo': 'Dakiti', 'artista': 'Bad Bunny', 'letra': "Baby, ya yo me enteré, se nota cuando me ve'\nAhí donde no ha' llega'o sabe' que yo te llevaré\nY dime qué quiere' beber, e' que tú ere' mi bebé\n¿Y de nosotro' quién va a hablar? Si no nos dejamo' ver\n\nY a vece' e' Dolce, a vece' Bulgari\nCuando te lo quito despué' de lo' partie'\nLas copa' de vino, las libra' de mari\nTú estás bien suelta, yo de safari\nTú muevе' el culo fenomenal\nPa' yo dеvorarte como animal\nSi no te ha' venío', yo te vo'a esperar\nEn mi cama y lo vo'a celebrar\n\nBaby, a ti no me opongo\nY siempre te lo pongo\nY si tú me tira', vamo' a nadar en lo hondo\nSi e' por mí te lo pongo\nDe septiembre hasta agosto\nA mí sin cojone' a lo que digan tu' amiga'\n[Estribillo: Bad Bunny & Jhay Cortez]\nYa yo me enteré, se nota cuando me ve'\nAhí donde no ha' llega'o sabe' que yo te llevaré\nY dime qué quiere' beber, e' que tú ere' mi bebé\n¿Y de nosotro' quién va a hablar? Si 